# 07 — ODE integration: Euler, RK4, and CVODES

**Objectives.** Separate a continuous model from a discrete propagation map, compare accuracy, and use an adaptive integrator. We solve $\dot x_1=x_2$, $\dot x_2=-x_1-0.2x_2+u$. Local error is $O(h^2)$ for Euler and $O(h^5)$ for RK4; accumulated global error is one order lower.


In [ ]:
import casadi as ca
import numpy as np
from casadi_tutorial.integration import euler_step, rk4_step
def rhs(x, u): return ca.vertcat(x[1], -x[0] - 0.2*x[1] + u[0])
x = ca.MX.sym('x', 2); u = ca.MX.sym('u', 1); h = 0.1
euler = ca.Function('euler', [x,u], [euler_step(rhs, x, u, h)])
rk4 = ca.Function('rk4', [x,u], [rk4_step(rhs, x, u, h)])
ode = {'x': x, 'p': u, 'ode': rhs(x,u)}
cvodes = ca.integrator('cvodes_step', 'cvodes', ode, 0.0, h, {'abstol':1e-10, 'reltol':1e-10})
reference = cvodes(x0=[1,0], p=[0])['xf']
print('Euler', euler([1,0],[0]), 'RK4', rk4([1,0],[0]), 'CVODES', reference)
assert float(ca.norm_2(rk4([1,0],[0]) - reference)) < float(ca.norm_2(euler([1,0],[0]) - reference))


A control is typically held piecewise constant over each interval. Integration failure can signal tight tolerances, stiffness, discontinuities, an invalid domain, or a bad model—not merely a need for more solver iterations. CVODES uses adaptive internal steps; the outer control grid is still yours.

**Common mistake:** silently mixing seconds and hours. **Recap:** validate a simple fixed-step map against an adaptive integrator before embedding it in an OCP. [Exercises](../exercises/03_integration.ipynb).
